# Nepali (Model L) - Easier Finetuning
This notebook trains the Nepali model on the easier, shorter dataset to overcome char-level tokenization bottlenecks.

In [ ]:
!pip install torch numpy matplotlib pandas requests

import os
import urllib.request

# Download the easy dataset from the repo
base_url = "https://huggingface.co/manyajain111/phase3-finetuned-models/resolve/main/data/ne_easy/"
os.makedirs("data/ne_easy", exist_ok=True)
for split in ['train.jsonl', 'val.jsonl', 'test.jsonl']:
    if not os.path.exists(f"data/ne_easy/{split}"):
        print(f"Downloading {split}...")
        urllib.request.urlretrieve(base_url + split, f"data/ne_easy/{split}")
print("Dataset ready!")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import json
import urllib.request
import matplotlib.pyplot as plt

urllib.request.urlretrieve("https://huggingface.co/manyajain111/phase3-finetuned-models/resolve/main/tokenizer_impl.py", "tokenizer_impl.py")
urllib.request.urlretrieve("https://huggingface.co/manyajain111/phase3-finetuned-models/resolve/main/scripts/generate_all_plots.py", "phase3_lib.py")

from tokenizer_impl import TokenizerWrapper
from phase3_lib import GPTLanguageModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
import os
if not os.path.exists("nepali_model_final.pt"):
    print("Downloading Nepali Phase 2 checkpoint...")
    urllib.request.urlretrieve("https://huggingface.co/manyajain111/lma-phase2/resolve/main/nepali/model_final.pt", "nepali_model_final.pt")

if not os.path.exists("nepali_tokenizer.json"):
    print("Downloading Nepali Phase 2 tokenizer...")
    urllib.request.urlretrieve("https://huggingface.co/manyajain111/lma-phase2/resolve/main/nepali/nepali_tokenizer.json", "nepali_tokenizer.json")

tok_l = TokenizerWrapper("nepali_tokenizer.json")
print(f"Nepali Vocab Size: {tok_l.vocab_size}")

model_l = GPTLanguageModel(vocab_size=tok_l.vocab_size).to(device)
sd = torch.load("nepali_model_final.pt", map_location=device)
model_l.load_state_dict(sd)
print("Model L loaded successfully.")

In [ ]:
class ReasoningDataset(Dataset):
    def __init__(self, jsonl_path, tokenizer, max_len=768):
        self.data = []
        with open(jsonl_path, 'r', encoding='utf-8') as f:
            for line in f:
                item = json.loads(line)
                text = f"{item['prompt']} {item['answer']}"
                tokens = tokenizer.encode(text)
                if len(tokens) <= max_len:
                    self.data.append(tokens)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        tokens = self.data[idx]
        x = torch.tensor(tokens[:-1], dtype=torch.long)
        y = torch.tensor(tokens[1:], dtype=torch.long)
        return x, y

def collate_fn(batch, pad_token_id=0):
    xs, ys = zip(*batch)
    max_len = max(len(x) for x in xs)
    x_padded = torch.stack([F.pad(x, (0, max_len - len(x)), value=pad_token_id) for x in xs])
    y_padded = torch.stack([F.pad(y, (0, max_len - len(y)), value=-100) for y in ys])
    return x_padded, y_padded

train_ds = ReasoningDataset("data/ne_easy/train.jsonl", tok_l, max_len=768)
val_ds = ReasoningDataset("data/ne_easy/val.jsonl", tok_l, max_len=768)
print(f"Train size: {len(train_ds)}, Val size: {len(val_ds)}")

In [ ]:
def evaluate_loss(model, dataloader, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1), ignore_index=-100)
            total_loss += loss.item()
    return total_loss / len(dataloader)

def train_finetune(model, epochs=3, lr=5e-6, batch_size=32):
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    
    history = {'train_loss': [], 'val_loss': []}
    for epoch in range(epochs):
        model.train()
        total_train = 0
        for i, (x, y) in enumerate(train_loader):
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1), ignore_index=-100)
            loss.backward()
            optimizer.step()
            total_train += loss.item()
            
            if (i+1) % 50 == 0:
                print(f"Epoch {epoch+1} | Step {i+1}/{len(train_loader)} | Loss: {loss.item():.4f}")
                
        avg_train = total_train / len(train_loader)
        avg_val = evaluate_loss(model, val_loader, device)
        print(f"=== Epoch {epoch+1} Summary: Train={avg_train:.4f}, Val={avg_val:.4f} ===")
        history['train_loss'].append(avg_train)
        history['val_loss'].append(avg_val)
        
    return history

# Lowered LR (5e-6) and 3 epochs to prevent the massive overfitting seen earlier!
history = train_finetune(model_l, epochs=3, lr=5e-6, batch_size=32)

In [ ]:
# Plot the loss curves immediately in the notebook
plt.figure(figsize=(8, 5))
plt.plot(range(1, 4), history['train_loss'], marker='o', label='Train Loss')
plt.plot(range(1, 4), history['val_loss'], marker='s', label='Val Loss')
plt.title('Nepali Easier Finetuning Loss (lr=5e-6)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.xticks([1, 2, 3])
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
def evaluate_exact_match(model, tokenizer, jsonl_path, num_samples=500):
    model.eval()
    correct_substring = 0
    correct_exact = 0
    samples = []
    with open(jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            if len(samples) >= num_samples: break
            samples.append(json.loads(line))
            
    for item in samples:
        prompt = item['prompt'] + " "
        target = item['label']
        
        tokens = tokenizer.encode(prompt)
        idx = torch.tensor([tokens], dtype=torch.long).to(device)
        
        # Generate up to 20 tokens
        with torch.no_grad():
            for _ in range(20):
                logits = model(idx)
                next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
                idx = torch.cat((idx, next_token), dim=1)
                
        generated = tokenizer.decode(idx[0].tolist()[len(tokens):])
        # Substring Match
        if target in generated:
            correct_substring += 1
        if generated.strip().startswith(item['answer'].strip()):
            correct_exact += 1
            
    print(f"\n--- Test Results (500 samples) ---")
    print(f"Exact Match Accuracy: {correct_exact/num_samples * 100:.2f}%")
    print(f"Substring Match Accuracy: {correct_substring/num_samples * 100:.2f}%")

evaluate_exact_match(model_l, tok_l, "data/ne_easy/test.jsonl", num_samples=500)

## Phase 3 Required Plots: Attention Comparison (Pre vs Post Finetuning)
To fulfill the Phase 3 requirement, we will load the base pre-trained model and compare its attention heatmaps to our newly finetuned model for Layer 0 and Layer 5.

In [ ]:
# Load the original un-finetuned model again for comparison
model_base = GPTLanguageModel(vocab_size=tok_l.vocab_size).to(device)
sd_base = torch.load("nepali_model_final.pt", map_location=device)
model_base.load_state_dict(sd_base)
model_base.eval()
model_l.eval()
print("Base model loaded for attention comparison!")

In [ ]:
from phase3_lib import plot_attention_comparison_multihead
import random

# Get a sample prompt from the test set
with open("data/ne_easy/test.jsonl", 'r', encoding='utf-8') as f:
    sample_item = json.loads(f.readline())

sample_text = f"{sample_item['prompt']} {sample_item['answer']}"
print(f"Plotting attention for: {sample_text}")

# Plot Layer 0 (Early)
plot_attention_comparison_multihead(
    tok=tok_l, 
    pre_model=model_base, 
    ft_model=model_l, 
    text=sample_text, 
    lang='nepali', 
    layer_idx=0, 
    is_early=True
)

# Plot Layer 5 (Late)
plot_attention_comparison_multihead(
    tok=tok_l, 
    pre_model=model_base, 
    ft_model=model_l, 
    text=sample_text, 
    lang='nepali', 
    layer_idx=5, 
    is_early=False
)